# Section 2.5 Reduced-Form AFS Strategy Validation

This notebook validates the flexible reduced-form strategy module. It uses the slow-moving liquidity heuristic by default and keeps fitted parameters external for later integration.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    if start.name == "notebooks":
        start = start.parent
    for candidate in [start] + list(start.parents):
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find project root")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ALPHA_INPUT = PROJECT_ROOT / "outputs" / "alphas" / "strategy_alpha_input_h5m_rho010_H5m.csv"
SCALING_PATH = PROJECT_ROOT / "outputs" / "scaling" / "scaling_factors_20d.csv"
BIN_DIR = PROJECT_ROOT / "data" / "binSamples"
OUT_DIR = PROJECT_ROOT / "outputs" / "strategy" / "reduced_form"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ALPHA_INPUT exists:", ALPHA_INPUT.exists())
print("SCALING_PATH exists:", SCALING_PATH.exists())

PROJECT_ROOT: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project
ALPHA_INPUT exists: True
SCALING_PATH exists: True


## 1. Course Formulas

Reduced-form liquidity: `lambda_t = lambda / sqrt(v_t)`, where `v_t` is rolling local market volume. Baseline target uses the slow-moving liquidity heuristic: `I*_t = 0.5 alpha_t - beta^{-1} mu_t`. The default trade translation uses the robust position formula `Q_t = I_t/lambda_t + sum beta I_t dt / lambda_t`.

In [2]:
from src.scaling_factors import compute_or_load_scaling_factors
from src.reduced_form_config import ReducedFormStrategyConfig
from src.reduced_form_strategy import (
    attach_scaling_factors,
    merge_market_volume_from_bins,
    run_reduced_form_strategy,
)
from src.reduced_form_metrics import (
    save_reduced_form_metrics,
    save_reduced_form_plots,
    compute_reduced_form_summary,
)
from src.reduced_form_validation import (
    validate_reduced_form_strategy_output,
    save_reduced_form_validation_report,
)

## 2. Load Alpha, Scaling, and Market Volume

In [3]:
if not ALPHA_INPUT.exists():
    raise FileNotFoundError(f"Missing alpha input: {ALPHA_INPUT}")

alpha_input = pd.read_csv(ALPHA_INPUT)
scaling = compute_or_load_scaling_factors(BIN_DIR, SCALING_PATH, force_recompute=False, window_days=20)

config = ReducedFormStrategyConfig(
    lambda_base=1.0,
    impact_half_life_minutes=5.0,
    volume_window_minutes=5.0,
    use_slow_moving_liquidity_heuristic=True,
    use_full_gamma_formula=False,
    translation_method="position_formula",
    scaling_factors_path=str(SCALING_PATH.relative_to(PROJECT_ROOT)),
)

if config.volume_col not in alpha_input.columns or "orderFlow" not in alpha_input.columns:
    alpha_input = merge_market_volume_from_bins(alpha_input, BIN_DIR, config)

strategy_input = attach_scaling_factors(alpha_input, scaling, config)
print("Alpha rows:", len(alpha_input))
print("Rows with scaling:", int(strategy_input["has_scaling"].sum()), "/", len(strategy_input))
display(strategy_input.head())
display(scaling.head())

Alpha rows: 200000
Rows with scaling: 0 / 200000


,date,time,timestamp,stock,mid,alpha_raw,alpha_for_strategy,future_return_h,valid_future_return,spread,depth,source_file,date_bin,trade,orderFlow,sigma,ADV,scaling_window_days,has_full_scaling_window,has_scaling
0,2019-01-02,09:30:10,2019-01-02 09:30:10,A,66.215,-0.000048,-0.000048,0.003020,True,0.285,300.0,bin201901.csv,2019-01-02,0,1300,NaN,NaN,0,False,False
1,2019-01-02,09:30:20,2019-01-02 09:30:20,A,66.335,-0.000114,-0.000050,0.001583,True,0.145,350.0,bin201901.csv,2019-01-02,0,-300,NaN,NaN,0,False,False
2,2019-01-02,09:30:30,2019-01-02 09:30:30,A,66.335,-0.000123,-0.000052,0.001583,True,0.145,600.0,bin201901.csv,2019-01-02,0,200,NaN,NaN,0,False,False
3,2019-01-02,09:30:40,2019-01-02 09:30:40,A,66.340,-0.000239,-0.000056,0.001733,True,0.140,500.0,bin201901.csv,2019-01-02,0,300,NaN,NaN,0,False,False
4,2019-01-02,09:31:00,2019-01-02 09:31:00,A,66.270,0.000156,-0.000046,0.003320,True,0.070,375.0,bin201901.csv,2019-01-02,0,0,NaN,NaN,0,False,False


,stock,date,trailing_px_vol,trailing_ADV,px_vol_current_day,daily_volume_current_day,scaling_window_days,has_full_scaling_window
0,A,2019-01-02,NaN,NaN,0.000393,217804.0,0,False
1,A,2019-01-03,NaN,NaN,0.000531,485020.0,1,False
2,A,2019-01-04,NaN,NaN,0.000367,244093.0,2,False
3,A,2019-01-07,NaN,NaN,0.000364,251175.0,3,False
4,A,2019-01-08,NaN,NaN,0.000378,181329.0,4,False


## 3. Run Reduced-Form Strategy

In [4]:
trades = run_reduced_form_strategy(strategy_input, config)
trades.to_csv(OUT_DIR / "reduced_form_strategy_trades.csv", index=False)

cols = [
    "date", "time", "stock", "mid", "alpha_t", "mu_t", "market_signed_volume",
    "local_volume_state_v", "lambda_t", "gamma_prime_t", "target_impact",
    "signed_volume_trade", "position_after", "impact_after_trade", "net_pnl",
    "skipped_due_to_missing_scaling",
]
display(trades[cols].head())

,date,time,stock,mid,alpha_t,mu_t,market_signed_volume,local_volume_state_v,lambda_t,gamma_prime_t,target_impact,signed_volume_trade,position_after,impact_after_trade,net_pnl,skipped_due_to_missing_scaling
0,2019-01-02,09:30:10,A,66.215,-0.000048,0.000000,0,1.000000e-12,1000000.0,0.0,-0.000024,0.0,0.0,0.0,0.0,True
1,2019-01-02,09:30:20,A,66.335,-0.000050,-0.000009,0,1.000000e-12,1000000.0,0.0,0.000040,0.0,0.0,0.0,0.0,True
2,2019-01-02,09:30:30,A,66.335,-0.000052,-0.000010,0,1.000000e-12,1000000.0,0.0,0.000046,0.0,0.0,0.0,0.0,True
3,2019-01-02,09:30:40,A,66.340,-0.000056,-0.000026,0,1.000000e-12,1000000.0,0.0,0.000157,0.0,0.0,0.0,0.0,True
4,2019-01-02,09:31:00,A,66.270,-0.000046,0.000029,0,1.000000e-12,1000000.0,0.0,-0.000230,0.0,0.0,0.0,-0.0,True


## 4. Validation Checks

In [5]:
checks = validate_reduced_form_strategy_output(trades, config)
checks_df = pd.DataFrame.from_dict(checks, orient="index")
display(checks_df)

impact_error = (trades["impact_after_trade"] - (trades["impact_before_trade"] + trades["lambda_t"] * trades["signed_volume_trade"])).abs().max()
position_error = (trades["position_after"] - (trades["position_before"] + trades["signed_volume_trade"])).abs().max()
print("Max impact dynamics error:", impact_error)
print("Max position dynamics error:", position_error)

,status,message
required_columns,PASS,missing=[]
no_critical_nans,PASS,"{'signed_volume_trade': 0, 'position_after': 0..."
dt_non_negative,PASS,min=0
beta_positive,PASS,min=0.139
volume_state_floor,PASS,min=1e-12
lambda_t_positive_finite,PASS,lambda_t finite and positive
gamma_prime_finite,PASS,gamma_prime_t finite
target_impact_finite,PASS,target impact finite
position_dynamics,PASS,max_error=0
impact_dynamics,PASS,max_error=0


Max impact dynamics error: 0.0
Max position dynamics error: 0.0


## 5. Metrics and Plots

In [6]:
daily_metrics, summary = save_reduced_form_metrics(trades, OUT_DIR)
save_reduced_form_plots(trades, daily_metrics, OUT_DIR)
save_reduced_form_validation_report(checks, trades, summary, config, OUT_DIR)

display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))
display(daily_metrics.head())

,value
total_gross_pnl,0.0
total_net_pnl,0.0
total_signed_impact_cost,0.0
total_quadratic_impact_cost,0.0
mean_daily_net_pnl,0.0
std_daily_net_pnl,0.0
daily_sharpe,NaN
annualized_sharpe,NaN
total_turnover_shares,0.0
total_turnover_notional,0.0


,date,gross_pnl,net_pnl,signed_impact_cost,quadratic_impact_cost,turnover_shares,turnover_notional,max_abs_position,max_abs_impact,max_participation_rate,mean_participation_rate,n_trades,n_stock_days,rows_with_scaling,rows_skipped,cumulative_daily_wealth,daily_drawdown
0,2019-01-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0,50,0,110382,0.0,0.0
1,2019-01-03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0,40,0,89528,0.0,0.0


In [7]:
for p in sorted(FIG_DIR.glob("reduced_form_*.png")):
    print(p.relative_to(PROJECT_ROOT), "exists=", p.exists())

outputs/strategy/reduced_form/figures/reduced_form_cumulative_wealth.png exists= True
outputs/strategy/reduced_form/figures/reduced_form_daily_net_pnl.png exists= True
outputs/strategy/reduced_form/figures/reduced_form_drawdown.png exists= True
outputs/strategy/reduced_form/figures/reduced_form_gross_vs_net_pnl.png exists= True
outputs/strategy/reduced_form/figures/reduced_form_lambda_t_histogram.png exists= True
outputs/strategy/reduced_form/figures/reduced_form_sample_strategy_path.png exists= True
outputs/strategy/reduced_form/figures/reduced_form_signed_volume_histogram.png exists= True
outputs/strategy/reduced_form/figures/reduced_form_target_vs_impact.png exists= True
outputs/strategy/reduced_form/figures/reduced_form_volume_state_histogram.png exists= True


## 6. Liquidity Diagnostics

In [8]:
diag_cols = ["local_volume_state_v", "lambda_t", "gamma_prime_t", "market_abs_volume", "participation_rate"]
display(trades[diag_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

,local_volume_state_v,lambda_t,gamma_prime_t,market_abs_volume,participation_rate
count,2.000000e+05,200000.000000,200000.000000,200000.000000,0.0
mean,8.589922e+03,1690.020576,-0.000040,295.856455,NaN
std,2.477586e+04,41074.959114,0.023518,1723.640666,NaN
min,1.000000e-12,0.001367,-3.000000,0.000000,NaN
1%,2.170000e+02,0.002938,-0.023659,0.000000,NaN
5%,5.900000e+02,0.005676,-0.008127,0.000000,NaN
50%,3.095000e+03,0.017975,0.000000,24.000000,NaN
95%,3.103905e+04,0.041169,0.008080,1000.000000,NaN
99%,1.158257e+05,0.067884,0.023132,4301.020000,NaN
max,5.353180e+05,1000000.000000,3.000000,310019.000000,NaN


## 7. Mechanical Sensitivity

This is not fitted-model calibration. It checks whether the code mechanics are stable across placeholder parameter choices.

In [9]:
rows = []
for lambda_base in [0.5, 1.0, 2.0]:
    for half_life in [1.0, 5.0, 30.0, 60.0]:
        for volume_window in [1.0, 5.0, 10.0]:
            cfg = ReducedFormStrategyConfig(
                lambda_base=lambda_base,
                impact_half_life_minutes=half_life,
                volume_window_minutes=volume_window,
                translation_method="position_formula",
                scaling_factors_path=str(SCALING_PATH.relative_to(PROJECT_ROOT)),
            )
            tr = run_reduced_form_strategy(strategy_input, cfg)
            rows.append({
                "lambda_base": lambda_base,
                "impact_half_life_minutes": half_life,
                "volume_window_minutes": volume_window,
                **compute_reduced_form_summary(tr),
            })

sensitivity = pd.DataFrame(rows)
sensitivity.to_csv(OUT_DIR / "reduced_form_sensitivity_summary.csv", index=False)
display(sensitivity[["lambda_base", "impact_half_life_minutes", "volume_window_minutes", "total_net_pnl", "total_turnover_shares", "max_abs_impact", "rows_with_scaling"]].head(20))

,lambda_base,impact_half_life_minutes,volume_window_minutes,total_net_pnl,total_turnover_shares,max_abs_impact,rows_with_scaling
0,0.5,1.0,1.0,0.0,0.0,0.0,0
1,0.5,1.0,5.0,0.0,0.0,0.0,0
2,0.5,1.0,10.0,0.0,0.0,0.0,0
3,0.5,5.0,1.0,0.0,0.0,0.0,0
4,0.5,5.0,5.0,0.0,0.0,0.0,0
5,0.5,5.0,10.0,0.0,0.0,0.0,0
6,0.5,30.0,1.0,0.0,0.0,0.0,0
7,0.5,30.0,5.0,0.0,0.0,0.0,0
8,0.5,30.0,10.0,0.0,0.0,0.0,0
9,0.5,60.0,1.0,0.0,0.0,0.0,0


## 8. Final Conclusion

Outputs are ready for section 2.7 stress-test integration when validation has no critical failures. Economic interpretation remains provisional until teammate fitted `lambda_base`, `beta`, and any stock-specific parameters are supplied.

In [10]:
print("Trades:", OUT_DIR / "reduced_form_strategy_trades.csv")
print("Daily metrics:", OUT_DIR / "reduced_form_daily_metrics.csv")
print("Summary metrics:", OUT_DIR / "reduced_form_summary_metrics.csv")
print("Validation report:", OUT_DIR / "reduced_form_validation_report.txt")
print("Sensitivity:", OUT_DIR / "reduced_form_sensitivity_summary.csv")
print("Critical failures:", checks_df[checks_df["status"].eq("FAIL")].index.tolist())

Trades: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/outputs/strategy/reduced_form/reduced_form_strategy_trades.csv
Daily metrics: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/outputs/strategy/reduced_form/reduced_form_daily_metrics.csv
Summary metrics: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/outputs/strategy/reduced_form/reduced_form_summary_metrics.csv
Validation report: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/outputs/strategy/reduced_form/reduced_form_validation_report.txt
Sensitivity: /Users/ulysse/Desktop/Studies/Imperial/Computing for finance/Price Impact/price-impact-project/outputs/strategy/reduced_form/reduced_form_sensitivity_summary.csv
Critical failures: []
